# AUTO_02_Customers_Ingestion — Production

This notebook implements incremental Customers ingestion using the same production pattern validated for Orders.

**Source:** `s3://olist-retail-project/raw/customers/`  
**Target:** `workspace.bronze.customers`  
**Checkpoint:** `s3://olist-retail-project/_checkpoints/customers_ingestion/`  
**Mode:** Auto Loader + append  
**Schema policy:** Strict (`failOnNewColumns`)  

Customers Bronze and Silver have different grains: Bronze retains source customer records, while Silver deduplicates to one row per `customer_unique_id`. Therefore this ingestion does **not** enforce `customer_unique_id` uniqueness in Bronze.

In [0]:
# ================================================================
# CELL 1 — AUTOMATED CUSTOMERS INGESTION CONFIGURATION
# ================================================================
# Defines the S3 source, dedicated checkpoint, and existing Bronze
# target. No filename or fixed row count is hardcoded.
# ================================================================

from pyspark.sql import functions as F

SOURCE_PATH = "s3://olist-retail-project/raw/customers/"

CHECKPOINT_PATH = (
    "s3://olist-retail-project/_checkpoints/customers_ingestion/"
)

BRONZE_TABLE = "workspace.bronze.customers"

print("Source     :", SOURCE_PATH)
print("Checkpoint :", CHECKPOINT_PATH)
print("Target     :", BRONZE_TABLE)


Source     : s3://olist-retail-project/raw/customers/
Checkpoint : s3://olist-retail-project/_checkpoints/customers_ingestion/
Target     : workspace.bronze.customers


In [0]:
# ================================================================
# CELL 2 — VERIFY EXISTING BRONZE TARGET
# ================================================================
# Confirms that the existing Bronze Customers table is available.
# The row count is informational only and is never a fixed baseline.
# ================================================================

if not spark.catalog.tableExists(BRONZE_TABLE):
    raise ValueError(
        f"Required Bronze table does not exist: {BRONZE_TABLE}"
    )

before_count = spark.table(BRONZE_TABLE).count()

print(f"Current Bronze Customers rows : {before_count:,}")
print(f"Bronze target verified         : {BRONZE_TABLE}")


Current Bronze Customers rows : 99,441
Bronze target verified         : workspace.bronze.customers


In [0]:
# ================================================================
# CELL 3 — CAPTURE BRONZE SCHEMA CONTRACT
# ================================================================
# Uses the existing Bronze schema instead of hardcoding data types.
# This keeps ingestion aligned with the current Bronze architecture.
# ================================================================

bronze_schema = spark.table(BRONZE_TABLE).schema

print("Bronze schema contract:")

for field in bronze_schema:
    print(
        f"  {field.name:<40} {field.dataType.simpleString()}"
    )


Bronze schema contract:
  customer_id                              string
  customer_unique_id                       string
  customer_zip_code_prefix                 int
  customer_city                            string
  customer_state                           string


In [0]:
# ================================================================
# CELL 4 — CREATE AUTO LOADER STREAM
# ================================================================
# Auto Loader discovers newly arriving CSV files in the Customers
# folder. Historical files are excluded because they are already
# represented in the existing Bronze table.
# ================================================================

customers_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .schema(bronze_schema)
        .option(
            "cloudFiles.schemaEvolutionMode",
            "failOnNewColumns"
        )
        .option(
            "cloudFiles.includeExistingFiles",
            "false"
        )
        .load(SOURCE_PATH)
)

print(
    "PASS — Auto Loader stream configured for new Customers files."
)


PASS — Auto Loader stream configured for new Customers files.


In [0]:
# ================================================================
# CELL 5 — INCREMENTAL WRITE TO BRONZE CUSTOMERS
# ================================================================
# Appends newly discovered records to the existing Bronze table.
# The checkpoint preserves Auto Loader progress between Job runs.
# availableNow=True lets the Job process currently available files
# and terminate; the next file-arrival trigger starts another run.
# ================================================================

query = (
    customers_stream
        .writeStream
        .outputMode("append")
        .option(
            "checkpointLocation",
            CHECKPOINT_PATH
        )
        .trigger(availableNow=True)
        .toTable(BRONZE_TABLE)
)

query.awaitTermination()

print(
    "PASS — Customers incremental ingestion completed successfully."
)


PASS — Customers incremental ingestion completed successfully.


In [0]:
# ================================================================
# CELL 6 — POST-INGESTION POPULATION VALIDATION
# ================================================================
# Reports the current Bronze population without assuming a fixed
# number of rows. Incoming files may contain any valid record count.
# ================================================================

after_count = spark.table(BRONZE_TABLE).count()

print(f"Current Bronze Customers rows : {after_count:,}")
print(f"Bronze target                 : {BRONZE_TABLE}")

print(
    "PASS — Bronze Customers table is available after ingestion."
)


Current Bronze Customers rows : 99,441
Bronze target                 : workspace.bronze.customers
PASS — Bronze Customers table is available after ingestion.


In [0]:
# ================================================================
# CELL 7 — POST-INGESTION SCHEMA VALIDATION
# ================================================================
# Confirms that the Bronze schema remains unchanged after ingestion.
# ================================================================

current_schema = spark.table(BRONZE_TABLE).schema

if current_schema != bronze_schema:
    raise ValueError(
        "Bronze Customers schema changed unexpectedly."
    )

print("PASS — Bronze Customers schema remains unchanged.")

print("Current Bronze columns:")

for field in current_schema:
    print(
        f"  {field.name:<40} {field.dataType.simpleString()}"
    )


PASS — Bronze Customers schema remains unchanged.
Current Bronze columns:
  customer_id                              string
  customer_unique_id                       string
  customer_zip_code_prefix                 int
  customer_city                            string
  customer_state                           string


In [0]:
# ================================================================
# CELL 8 — BRONZE CUSTOMER IDENTITY QUALITY VALIDATION
# ================================================================
# customer_unique_id is the customer business key used by Silver.
# It may repeat in Bronze and therefore MUST NOT be checked for
# uniqueness here. The source customer_id is checked for duplicates
# to detect accidental duplicate source records.
# ================================================================

customers = spark.table(BRONZE_TABLE)

duplicate_customer_ids = (
    customers
        .filter(F.col("customer_id").isNotNull())
        .groupBy("customer_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
)

null_customer_unique_ids = (
    customers
        .filter(F.col("customer_unique_id").isNull())
        .count()
)

blank_customer_unique_ids = (
    customers
        .filter(F.trim(F.col("customer_unique_id")) == "")
        .count()
)

print(
    f"Duplicate customer_id values    : {duplicate_customer_ids:,}"
)
print(
    f"NULL customer_unique_id values  : {null_customer_unique_ids:,}"
)
print(
    f"Blank customer_unique_id values : {blank_customer_unique_ids:,}"
)

if duplicate_customer_ids != 0:
    raise ValueError(
        "Bronze Customers quality check failed: "
        "duplicate customer_id values detected."
    )

if null_customer_unique_ids != 0:
    raise ValueError(
        "Bronze Customers quality check failed: "
        "NULL customer_unique_id values detected."
    )

if blank_customer_unique_ids != 0:
    raise ValueError(
        "Bronze Customers quality check failed: "
        "blank customer_unique_id values detected."
    )

print(
    "PASS — Bronze Customers identity validation passed."
)


Duplicate customer_id values    : 0
NULL customer_unique_id values  : 0
Blank customer_unique_id values : 0
PASS — Bronze Customers identity validation passed.


In [0]:
# ================================================================
# CELL 9 — SILVER-GRAIN COMPATIBILITY CHECK
# ================================================================
# This does NOT write Silver. It verifies that Bronze contains valid
# customer identities for the existing Silver deduplication logic.
# Silver is expected to reduce Bronze to one row per customer_unique_id.
# ================================================================

bronze_customer_identities = (
    customers
        .select("customer_unique_id")
        .distinct()
        .count()
)

bronze_customer_rows = customers.count()

print(
    f"Bronze customer rows                 : {bronze_customer_rows:,}"
)
print(
    f"Distinct Bronze customer identities  : "
    f"{bronze_customer_identities:,}"
)

if bronze_customer_identities > bronze_customer_rows:
    raise ValueError(
        "Customers grain validation failed: "
        "distinct customer identities exceed Bronze row count."
    )

print(
    "PASS — Bronze grain is compatible with Silver customer-level deduplication."
)


Bronze customer rows                 : 99,441
Distinct Bronze customer identities  : 96,096
PASS — Bronze grain is compatible with Silver customer-level deduplication.


In [0]:
# ================================================================
# CELL 10 — FINAL INGESTION STATUS
# ================================================================
# Produces a concise status message for Databricks Job logs.
# ================================================================

print("=" * 70)
print("CUSTOMERS AUTOMATED INGESTION — SUCCESS")
print("=" * 70)
print(f"Source       : {SOURCE_PATH}")
print(f"Target       : {BRONZE_TABLE}")
print(f"Current rows : {after_count:,}")
print("Mode         : Incremental append")
print("File handling: Auto Loader")
print("Schema mode  : Strict")
print("=" * 70)


CUSTOMERS AUTOMATED INGESTION — SUCCESS
Source       : s3://olist-retail-project/raw/customers/
Target       : workspace.bronze.customers
Current rows : 99,441
Mode         : Incremental append
File handling: Auto Loader
Schema mode  : Strict
